# How much oil is left?

Reproduce the article’s opening example from the [EIA and OPEC source extracts](../data/README.md). All numerical cells use the repository’s Python standard-library models. Run the cells in order.

In [1]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/oil_evidence/models.py").is_file())
sys.path.insert(0, str(root / "src"))
data = root / "data"
from oil_evidence.models import read_integers, us_history, world_history

## A decade of US production

Start at the **end** of 2014. US crude oil and lease condensate production from 2015 onward is the flow to compare with this opening stock.

In [2]:
us = list(us_history(read_integers(data / "us_reserves.csv")))
print("Year    Closing reserves    Produced since end-2014  (billion barrels)")
for row in us:
    print(f"{row['year']}    {row['reserves_mmbbl']/1000:>9.3f}          {row['cumulative_since_base_mmbbl']/1000:>9.3f}")

Year    Closing reserves    Produced since end-2014  (billion barrels)
2014       39.933              0.000
2015       35.230              3.427
2016       35.213              6.650
2017       41.990             10.051
2018       47.053             14.035
2019       47.172             18.515
2020       38.212             22.669
2021       44.418             26.798
2022       48.321             31.166
2023       46.422             35.871
2024       45.951             40.677


In [3]:
start, end = us[0], us[-1]
opening = start["reserves_mmbbl"]
closing = end["reserves_mmbbl"]
produced = end["cumulative_since_base_mmbbl"]
added = end["implied_net_additions_since_base_mmbbl"]
assert opening + added - produced == closing
print(f"Opening + net additions − production = closing (billion barrels)")
print(f"{opening/1000:.3f} + {added/1000:.3f} − {produced/1000:.3f} = {closing/1000:.3f}")

Opening + net additions − production = closing (billion barrels)
39.933 + 46.695 − 40.677 = 45.951


![US reserves and cumulative production](../outputs/us-reserves.png)

Net additions include extensions, discoveries, revisions and other reported changes. This balance cannot tell us how many barrels were newly discovered.

## The world ratio

OPEC reports year-end crude reserves in million barrels and average production in thousand barrels per day. Use the actual days in each year, including leap years.

In [4]:
world = list(world_history(read_integers(data / "world_crude.csv")))
for row in world:
    print(f"{row['year']}   {row['calendar_days']} days   {row['production_mmbbl']/1000:.3f} bn barrels produced   {row['rp_years']:.2f} years")

2020   366 days   25.191 bn barrels produced   61.28 years
2021   365 days   25.343 bn barrels produced   61.50 years
2022   365 days   26.616 bn barrels produced   58.74 years
2023   365 days   26.773 bn barrels produced   58.44 years
2024   366 days   26.566 bn barrels produced   58.98 years


Try holding the reserve balance fixed and reducing annual output by 10%. The ratio gets larger immediately. What changed physically on the day the production rate changed? The US survey and world bulletin have different product coverage. Use each series within its own definition.